In [7]:
import json
import time
import numpy as np
import os
from dotenv import load_dotenv

load_dotenv() 

PROVIDER = os.environ['PROVIDER']
API_KEY = os.environ['API_KEY']
NUM_PAIRS = 200

with open("dpo_pairs.json") as f:
    data = json.load(f)
pairs = data["pairs"][:NUM_PAIRS]

print(f"Loaded {len(pairs)} pairs for evaluation")
print(f"Example pair:")
print(f"Prompt: {pairs[0]['prompt'][:80]}...")
print(f"Chosen reward: {pairs[0]['reward_chosen']:.3f}")
print(f"Rejected reward: {pairs[0]['reward_rejected']:.3f}")

Loaded 200 pairs for evaluation
Example pair:
Prompt: Process:  - An owl leaves his nest - The owl flys out into the night - The owl l...
Chosen reward: 1.305
Rejected reward: -0.883


In [8]:
if PROVIDER == "anthropic":
    from anthropic import Anthropic
    client = Anthropic(api_key=API_KEY)
    MODEL = "claude-opus-4-6"

JUDGE_PROMPT = """You are an impartial judge. Given a user question and two assistant responses (A and B), decide which response is better.

Consider: helpfulness, accuracy, relevance, and clarity.

Reply with ONLY the single letter "A" or "B". No explanation.

User question:
{prompt}

Response A:
{response_a}

Response B:
{response_b}"""


def judge(prompt, response_a, response_b):
    text = JUDGE_PROMPT.format(prompt=prompt, response_a=response_a, response_b=response_b)

    if PROVIDER == "anthropic":
        resp = client.messages.create(
            model=MODEL, max_tokens=16,
            messages=[{"role": "user", "content": text}],
        )
        answer = resp.content[0].text.strip()
    else:
        resp = client.chat.completions.create(
            model=MODEL, max_tokens=16,
            messages=[{"role": "user", "content": text}],
        )
        answer = resp.choices[0].message.content.strip()

    return answer

test = judge(pairs[0]["prompt"], pairs[0]["chosen"], pairs[0]["rejected"])
print(f"Test judge call returned: '{test}'")

Test judge call returned: 'B'


In [9]:
# A=chosen, B=rejected
forward_results = []

for i, pair in enumerate(pairs):
    verdict = judge(pair["prompt"], pair["chosen"], pair["rejected"])
    forward_results.append(verdict)

    if (i + 1) % 50 == 0:
        valid = [v for v in forward_results if v != "SKIP"]
        agree = sum(1 for v in valid if v == "A")
        print(f"[{i+1}/{len(pairs)}] Agreement with RM (A=chosen wins): {agree}/{len(valid)} ({agree/len(valid)*100:.1f}%) skipped={sum(1 for v in forward_results if v == 'SKIP')}")

valid_forward = [v for v in forward_results if v != "SKIP"]
forward_agreement = sum(1 for v in valid_forward if v == "A") / len(valid_forward) if valid_forward else 0
print(f"\nForward pass done. Agreement with RM: {forward_agreement*100:.1f}% ({len(valid_forward)} valid, {len(forward_results)-len(valid_forward)} skipped)")

[50/200] Agreement with RM (A=chosen wins): 18/50 (36.0%) skipped=0
[100/200] Agreement with RM (A=chosen wins): 37/100 (37.0%) skipped=0
[150/200] Agreement with RM (A=chosen wins): 60/150 (40.0%) skipped=0
[200/200] Agreement with RM (A=chosen wins): 83/200 (41.5%) skipped=0

Forward pass done. Agreement with RM: 41.5% (200 valid, 0 skipped)


In [10]:
# A=rejected, B=chosen 
reverse_results = []

for i, pair in enumerate(pairs):
    verdict = judge(pair["prompt"], pair["rejected"], pair["chosen"])
    reverse_results.append(verdict)

    if (i + 1) % 50 == 0:
        valid = [v for v in reverse_results if v != "SKIP"]
        agree = sum(1 for v in valid if v == "B")
        print(f"[{i+1}/{len(pairs)}] Agreement with RM (B=chosen wins): {agree}/{len(valid)} ({agree/len(valid)*100:.1f}%) skipped={sum(1 for v in reverse_results if v == 'SKIP')}")

valid_reverse = [v for v in reverse_results if v != "SKIP"]
reverse_agreement = sum(1 for v in valid_reverse if v == "B") / len(valid_reverse) if valid_reverse else 0
print(f"\nReverse pass done. Agreement with RM: {reverse_agreement*100:.1f}% ({len(valid_reverse)} valid, {len(reverse_results)-len(valid_reverse)} skipped)")

[50/200] Agreement with RM (B=chosen wins): 18/50 (36.0%) skipped=0
[100/200] Agreement with RM (B=chosen wins): 35/100 (35.0%) skipped=0
[150/200] Agreement with RM (B=chosen wins): 53/150 (35.3%) skipped=0
[200/200] Agreement with RM (B=chosen wins): 73/200 (36.5%) skipped=0

Reverse pass done. Agreement with RM: 36.5% (200 valid, 0 skipped)


In [12]:
valid_fwd = [v for v in forward_results if v != "SKIP"]
valid_rev = [v for v in reverse_results if v != "SKIP"]

forward_a_rate = sum(1 for v in valid_fwd if v == "A") / len(valid_fwd) if valid_fwd else 0
reverse_a_rate = sum(1 for v in valid_rev if v == "A") / len(valid_rev) if valid_rev else 0

position_bias = forward_a_rate - (1 - reverse_a_rate)
overall_agreement = (forward_agreement + reverse_agreement) / 2

consistent = sum(
    1 for fwd, rev in zip(forward_results, reverse_results)
    if fwd == "A" and rev == "B"
)
consistent_rate = consistent / sum(
    1 for fwd, rev in zip(forward_results, reverse_results)
    if fwd != "SKIP" and rev != "SKIP"
) if consistent else 0

results = {
    "forward_agreement": forward_agreement,
    "reverse_agreement": reverse_agreement,
    "overall_agreement": overall_agreement,
    "position_bias": position_bias,
    "consistent_agreement": consistent_rate,
    "forward_a_rate": forward_a_rate,
    "reverse_a_rate": reverse_a_rate,
    "num_pairs": len(pairs),
    "num_skipped_forward": len(forward_results) - len(valid_fwd),
    "num_skipped_reverse": len(reverse_results) - len(valid_rev),
    "model": MODEL,
}

print("=" * 50)
print(f"Overall agreement with RM: {overall_agreement*100:.1f}%")
print(f"Consistent (both dirs): {consistent_rate*100:.1f}%")
print(f"Position bias: {position_bias:+.3f}")
print(f"Forward A-rate: {forward_a_rate*100:.1f}%")
print(f"Reverse A-rate: {reverse_a_rate*100:.1f}%")
print(f"(0 = no bias, + = prefers position A)")

with open("llm_critic_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved to llm_critic_results.json")

Overall agreement with RM: 39.0%
Consistent (both dirs): 27.0%
Position bias: -0.195
Forward A-rate: 41.5%
Reverse A-rate: 39.0%
(0 = no bias, + = prefers position A)
Saved to llm_critic_results.json
